In [1]:
from scripts.conf_file_finding import try_find_conf_file
try_find_conf_file()

Local configuration file found !!, no need to run the configuration (unless configuration has changed)


In [13]:
import pandas as pd
import datajoint as dj
import pathlib

In [5]:
import u19_pipeline.ephys_pipeline as ep
import u19_pipeline.recording as recording
import u19_pipeline.recording_process as rp
import u19_pipeline.lab as lab

In [9]:
df_path = pd.DataFrame((lab.Path & "global_path='/braininit'").fetch('system', 'local_path',as_dict=True))

linux_path = df_path.loc[df_path['system']=='linux','local_path'].values[0]
windows_path = df_path.loc[df_path['system']=='windows','local_path'].values[0]
mac_path = df_path.loc[df_path['system']=='mac','local_path'].values[0]

In [37]:
linux_path

'/mnt/cup/braininit'

In [38]:
windows_path

'//cup.pni.princeton.edu/braininit'

In [39]:
mac_path

'/Volumes/braininit'

In [34]:
rec_query = dict()
rec_query['recording_modality'] = 'electrophysiology'
rec_query_2 = "recording_id > 230"

rec_data = pd.DataFrame((recording.Recording * rp.Processing & rec_query & rec_query_2).fetch('recording_process_post_path',as_dict=True))
rec_data

,recording_process_post_path
0,jk8386/jk8386_jknpx3/20230827_g0/jknpx3_082720...
1,jk8386/jk8386_jknpx3/20230830_g0/jknpx3_083020...
2,jk8386/jk8386_jknpx3/20230831_g0/jknpx3_083120...
3,jk8386/jk8386_jknpx3/20230904_g0/jknpx3_090420...
4,jk8386/jk8386_jknpx3/20230905_g0/jknpx3_090520...
...,...
550,emdia/emdia_nicky/20260408_g0/nicky_20260408_g...
551,emdia/emdia_nicky/20260412_g0/nicky_20260412_g...
552,emdia/emdia_nicky/20260407_g0/nicky_20260407_g...
553,jk8386/jk8386_jknpx7/20260205_g0/jknpx7_202602...


In [36]:

for i in range(rec_data.shape[0]):
    proc_dir = rec_data.loc[i,'recording_process_post_path']
    output_dir = pathlib.Path(dj.config['custom']['ephys_root_data_dir'][1],proc_dir)

    kilo_output = output_dir.glob('*kilosort*')
    for kilo_dir in kilo_output:
        params_file = pathlib.Path(kilo_dir,'params.py')
        if params_file.is_file():
            
            with open(params_file.as_posix(), "r") as file:
                params_text = file.read()
            
            try:
                windows_params = params_text.replace(linux_path,windows_path)
                windows_param_file = pathlib.Path(kilo_dir,'win_params.py')
                with open(windows_param_file, "w") as file:
                    file.write(windows_params)

                mac_params = params_text.replace(linux_path,mac_path)
                mac_param_file = pathlib.Path(kilo_dir,'mac_params.py')
                with open(mac_param_file, "w") as file:
                    file.write(mac_params)
            except Exception as e:
                print(e)




[Errno 13] Permission denied: '/mnt/cup/braininit/Data/Processed/electrophysiology/jk8386/jk8386_jknpx3/20230908_g0/jknpx3_09082023_g0/jknpx3_09082023_g0_imec0/job_id_597/kilosort_output/win_params.py'
[Errno 13] Permission denied: '/mnt/cup/braininit/Data/Processed/electrophysiology/jk8386/jk8386_jknpx3/20230909_g0/jknpx3_09092023_g0/jknpx3_09092023_g0_imec0/job_id_598/kilosort_output/win_params.py'


In [31]:
kilo_dir

PosixPath('/mnt/cup/braininit/Data/Processed/electrophysiology/jk8386/jk8386_jknpx6/20251204_g0/jknpx6_20251204_g0/jknpx6_20251204_g0_imec0/job_id_998/kilosort4_output')